In [ ]:

# ============ IMPORTS ============
import pandas as pd
import numpy as np
import mlflow
from src.calculate_ml_metrics import calculate_ml_metrics
from src.calculate_trade_metrics import calculate_trade_metrics
import quantstats as qs
import matplotlib.pyplot as plt

# =========== CONFIGURATION ============
feature_set = "confirmed" # "all" or "confirmed" -  features confirmed by boruta algorithm
window = 7
model_names = ["baseline_majority"]

# "baseline_stratified"
# "baseline_buy_hold", 
# "log_reg"
# "random_forest"
# "svc"
# "xgboost"

# ======================================

# ============ LABEL REMAPPING ============
label_map   = {-1: 0, 0: 1, 1: 2}
label_unmap = { 0: -1, 1: 0, 2: 1}

# =========== MLFLOW SETUP ============
experiment  = mlflow.get_experiment_by_name("07_triple_barrier_classification")
experiment_id = experiment.experiment_id

# =========== LIST PREVIOUS RUNS ============
ml_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                            filter_string = "tags.Hyperpar_optimised_for = 'Balanced_F1_macro'", #tags.Hyperpar_optimised_for = 'GT_Score', status = 'FINISHED
                            order_by = ['start_time DESC'])

# ========== LOAD LATEST MODELS ==========
latest_run_df = ml_runs.groupby('tags.mlflow.runName').first()

# =========== INITIALISE DICTIONARIES ============
models = {}
train_predictions = {}
test_predictions = {}
selected_features = {}
# =========== LOAD MODELS AND PREDICTIONS ============
for model_name in model_names:
    run_name = f"{model_name}_{feature_set}_features"
    try:
        run_id = latest_run_df.loc[run_name, "run_id"]
        models[model_name] = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

        train_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"07_final_ml_train_data_{model_name}_{feature_set}.pkl"
        )
        test_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"07_final_ml_test_data_{model_name}_{feature_set}.pkl"
        )
        
        train_predictions[model_name] = pd.read_pickle(train_path)
        test_predictions[model_name]  = pd.read_pickle(test_path)

    except KeyError:
        print(f"No finished run found for '{run_name}' — skipping.")
        
train_df = train_predictions[model_name]
test_df = test_predictions[model_name]
y_train = train_df[f"Label_{window}day"]
train_pred = train_df[f"pred_{model_name}"]
train_proba = train_df[f"proba_{model_name}"]
y_test = test_df[f"Label_{window}day"]
test_pred = test_df[f"pred_{model_name}"]
test_proba = test_df[f"proba_{model_name}"]